# Determining feature importance. 

**Working Example.** Copy this file, rename it, and modify *your* copy.
See earlier notebooks for more information. 

- Author: Denise Case, Kim Hummel
- Date: 2026-06
- Dataset: Diabetes_data
- Target: feature with most importance 


## Overview

In the last project, in phase 4, I used data on people with diabetes. I wanted to reuse this data, but I don't know that I used the most importance features. So I'm going to run part of the module 5 phase 4 project again to get better information about what I should used in my server. 

I will do a rainforest ensemble and then graph feature importance. 

## Section 1. Project Setup and Imports

In [18]:
# === Section 1a. DECLARE IMPORTS ===

from importlib.metadata import version  # to verify
import logging  # for type hinting
import platform  # to verify
from typing import Final  # for type hinting

from datafun_toolkit.logger import get_logger, log_header
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# === Section 1b. CONFIGURE LOGGER ONCE PER NOTEBOOK ===

LOG: logging.Logger = get_logger("M05", level="DEBUG")
log_header(LOG, "M05")


# === Section 1c. USE THE LOGGER TO VERIFY IMPORTS ===

# If any do NOT return a version number, then that package is not installed correctly.
# Check your pyproject.toml and re-run environment setup commands.

LOG.info("Confirming installation:")
LOG.info(f"  python:       {platform.python_version()}")
LOG.info(f"  pandas:       {version('pandas')}")
LOG.info(f"  numpy:        {version('numpy')}")
LOG.info(f"  scikit-learn: {version('scikit-learn')}")
LOG.info(f"  seaborn:      {version('seaborn')}")
LOG.info(f"  matplotlib:   {version('matplotlib')}")


# === Section 1d. SET PANDAS DISPLAY CONFIGURATION (helps in notebooks) ===

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

2026-08-09 13:34:09 | INFO | M05 | === RUN START ===
2026-08-09 13:34:09 | INFO | M05 | project=M05
2026-08-09 13:34:09 | INFO | M05 | repo_dir=ml-06-serving
2026-08-09 13:34:09 | INFO | M05 | python=3.14.3
2026-08-09 13:34:09 | INFO | M05 | os=Windows 11
2026-08-09 13:34:09 | INFO | M05 | shell=powershell
2026-08-09 13:34:09 | INFO | M05 | cwd=project06
2026-08-09 13:34:09 | INFO | M05 | github_actions=False
2026-08-09 13:34:09 | INFO | M05 | Confirming installation:
2026-08-09 13:34:09 | INFO | M05 |   python:       3.14.3
2026-08-09 13:34:09 | INFO | M05 |   pandas:       3.0.5
2026-08-09 13:34:09 | INFO | M05 |   numpy:        2.5.1
2026-08-09 13:34:09 | INFO | M05 |   scikit-learn: 1.9.0
2026-08-09 13:34:09 | INFO | M05 |   seaborn:      0.13.2
2026-08-09 13:34:09 | INFO | M05 |   matplotlib:   3.11.1


## Section 2. Load and Prepare the Data

In [19]:
# === Section 2. Load and Prepare the Data ===


DATASET_NAME = "diabetes_data.csv"

LOG.info(f"Loading dataset: {DATASET_NAME}")
df: pd.DataFrame = pd.read_csv(DATASET_NAME)
LOG.info(f"Loaded: {df.shape[0]} rows (instances), {df.shape[1]} columns")

# CUSTOM: ANALYST CHOICE - the categorical target to classify.
TARGET_COL: Final[str] = "outcome"

# CUSTOM: Numeric features used to predict the target.
FEATURE_COLS: Final[list[str]] = [
    "pregnancies",
    "glucose",
    "blood_pressure",
    "skin_thickness",
    "insulin",
    "BMI",
    "diabetes_pedigree_function",
    "age",
]

required: list[str] = [TARGET_COL, *FEATURE_COLS]
df_model: pd.DataFrame = df.dropna(subset=required).copy()

LOG.info(f"Original rows: {df.shape[0]}")
LOG.info(f"Model rows:    {df_model.shape[0]}")
LOG.info(f"Classes in target '{TARGET_COL}': {sorted(df_model[TARGET_COL].unique())}")
LOG.debug(f"Class counts:\n{df_model[TARGET_COL].value_counts()}")

2026-08-09 13:34:14 | INFO | M05 | Loading dataset: diabetes_data.csv
2026-08-09 13:34:14 | INFO | M05 | Loaded: 768 rows (instances), 9 columns
2026-08-09 13:34:14 | INFO | M05 | Original rows: 768
2026-08-09 13:34:14 | INFO | M05 | Model rows:    768
2026-08-09 13:34:14 | INFO | M05 | Classes in target 'outcome': [np.int64(0), np.int64(1)]
2026-08-09 13:34:14 | DEBUG | M05 | Class counts:
outcome
0    500
1    268
Name: count, dtype: int64


## Section 3. Split into Train and Test

We will stratify the data between people with diabetes (0) and people without diabetes (1). 

In [20]:
# === Section 3. Split into Train and Test ===

# CUSTOM: ANALYST CHOICE

# Reproducibility for the split and the model.
# Pick any integer, but keep it the same to get the same results.
RANDOM_STATE: Final[int] = 42

"""Build X and y, then split into stratified train and test sets.

WHY: Scoring on held-out data is the only
honest estimate of performance on new data.
Stratifying preserves the class balance in both splits.
"""
X: pd.DataFrame = df_model[FEATURE_COLS]
y: pd.Series = df_model[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
LOG.info(f"Train instances: {len(X_train)}")
LOG.info(f"Test instances: {len(X_test)}")

2026-08-09 13:34:26 | INFO | M05 | Train instances: 614
2026-08-09 13:34:26 | INFO | M05 | Test instances: 154


In [21]:
# === Section 4. Fit a Single Model and Random Tree Ensemble ===

"""Fit one single model and one ensemble
on the same training data.

"""
LOG.info("Fitting single tree + two ensembles")

modelA = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
modelA.fit(X_train, y_train)
LOG.debug("  fitted: single_tree")

modelB = RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE)
modelB.fit(X_train, y_train)
LOG.debug("  fitted: random_forest")

modelC = VotingClassifier(
    estimators=[
        ("DT", DecisionTreeClassifier()),
        ("SVM", SVC(probability=True)),
        ("NN", MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000)),
    ],
    voting="soft",
)
modelC.fit(X_train, y_train)
LOG.debug("  fitted: voting classifier")

2026-08-09 13:34:30 | INFO | M05 | Fitting single tree + two ensembles
2026-08-09 13:34:30 | DEBUG | M05 |   fitted: single_tree
2026-08-09 13:34:31 | DEBUG | M05 |   fitted: random_forest
c:\Repos\Applied_Machine_Learning\ml-06-serving\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
2026-08-09 13:34:31 | DEBUG | M05 |   fitted: voting classifier


## Section 5. Compare on Held-Out Data

In [ ]:
# === Section 5. Compare on Held-Out Data ===

"""Score every model on the same test set and chart the results.

WHY: The only fair comparison is on data none of them trained on.

Code only reports numbers.

Whether an ensemble's gain
is large enough to justify its cost
depends on this specific problem,
and requires analyst judgment.
"""

# score each model on the held-out test set
# accuracy_score returns a numpy scalar - wrap in float() for compatibility
acc_single_tree: float = float(accuracy_score(y_test, modelA.predict(X_test)))
acc_random_forest: float = float(accuracy_score(y_test, modelB.predict(X_test)))
acc_voting_classifier: float = float(accuracy_score(y_test, modelC.predict(X_test)))

LOG.info(f"  {'single_tree':18s} test accuracy = {acc_single_tree:.3f}")
LOG.info(f"  {'random_forest':18s} test accuracy = {acc_random_forest:.3f}")
LOG.info(f"  {'voting_classifier':18s} test accuracy = {acc_voting_classifier:.3f}")

# build a dict for charting
# The keys are the model names,
# and the values are the test accuracies.
scores: dict[str, float] = {
    "single_tree": acc_single_tree,
    "random_forest": acc_random_forest,
    "voting_classifier": acc_voting_classifier,
}

# start a new figure
plt.figure()

# create a bar chart of the scores
x_keys = list(scores.keys())
y_values = list(scores.values())

# create a bar chart with seaborn (uses matplotlib under the hood)
sns.barplot(x=x_keys, y=y_values, palette="viridis")

# set y-axis limits to [0, 1] since accuracy is a percentage
plt.ylim(0, 1)

# label the axes and title
plt.xlabel("model")
plt.ylabel("test accuracy")
plt.title("Single model vs. ensembles (test set)")

# slight rotation for readability (optional)
plt.xticks(rotation=15)

# show the plot
plt.show()

# find the best model by name (the one with the highest test accuracy)
best: str = max(scores, key=scores.get)  # type: ignore[arg-type]
LOG.info(f"Best model: {best} with test accuracy = {scores[best]:.3f}")

## Section 6. Which Features Did the Forest Rely On?

In [ ]:
# === Section 6. Feature Importances (Random Forest) ===


"""Chart the random forest's feature importances.

WHY: a forest can still report which features it leaned on.
Importances are a clue, not proof of causation.

"""
# get the feature importances
# from the fitted random forest model
importances: np.ndarray = modelB.feature_importances_

# argsort returns indices that would sort the array ascending
# [::-1] reverses it so the most important feature comes first
order: np.ndarray = np.argsort(importances)[::-1]
LOG.info("Random forest feature importances (high to low):")
for i in order:
    LOG.info(f"  {FEATURE_COLS[i]:20s} {importances[i]:.3f}")

# start a new figure
plt.figure()

# create x and y values for the bar chart, ordered from high to low importance
x_values = [FEATURE_COLS[i] for i in order]
y_values = importances[order]

# create a bar chart of the importances, ordered from high to low
sns.barplot(x=x_values, y=y_values)

# label the axes and title
plt.xlabel("feature")
plt.ylabel("importance")
plt.title("Random forest feature importances")

# slight rotation for readability (optional)
plt.xticks(rotation=20)

# show the plot
plt.show()

## Section 7. Summary and Next Steps

The random forest performed better than the voting classifier so I will stick with the random forest in the project. 

Initially I was going to use the top 4 features, but upon reflection, the top three features most people or doctors offices will have readily available whereas a diabetes pedigree function is harder information to come by. So I will just use the top three features in my server. 